In [9]:
from pathlib import Path
import os, glob, re
import numpy as np
import pandas as pd
from biopandas.pdb import PandasPdb
import MDAnalysis as mda

# =========================
# Paths
# =========================
RCSB_PROTEIN_DIR = Path("CLR-PDB-matched-to-opm_ids")
UNLABELED_DIR = Path("CLR-Unlabeled-Distinct-matched-to-opm_ids")
OPM_DIR = Path("OPM_PDB")

def get_protein_name(filename):
    basename = os.path.basename(filename)
    match = re.match(r'([a-zA-Z0-9]{4})', basename)
    return match.group(1).upper() if match else None


def get_mode_index(filename):
    basename = os.path.basename(filename)
    match = re.search(r'mode_(\d+)', basename)
    return int(match.group(1)) if match else None


def natural_sort_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', str(s))]


def grid_list(atom_df):
    return list(zip(atom_df['x_coord'], atom_df['y_coord'], atom_df['z_coord']))


def filtering_proteins(atom_df, ligand_grid, radius=5.0):
    atom_coords = atom_df[['x_coord', 'y_coord', 'z_coord']].values
    filtered_atoms = set()

    for x, y, z in ligand_grid:
        dist_sq = (
            (atom_coords[:, 0] - x) ** 2 +
            (atom_coords[:, 1] - y) ** 2 +
            (atom_coords[:, 2] - z) ** 2
        )
        mask = dist_sq <= radius ** 2
        filtered_atoms.update(atom_df.index[mask])

    return atom_df.loc[list(filtered_atoms)].copy()


def read_protein_atoms(pdb_path):
    ppdb = PandasPdb().read_pdb(str(pdb_path))
    atoms = ppdb.df["ATOM"].copy()
    atoms = atoms[~atoms["atom_name"].str.startswith("H")].copy()
    return atoms


def read_clr_atoms(pdb_path):
    ppdb = PandasPdb().read_pdb(str(pdb_path))
    hetatm = ppdb.df["HETATM"].copy()
    clr = hetatm[hetatm["residue_name"] == "CLR"].copy()
    return clr


def atom_match_key(df):
    """
    Match RCSB atoms to OPM atoms by residue number, chain, and atom subtype/name.
    """
    return list(zip(
        df["residue_number"].astype(int),
        df["chain_id"].astype(str),
        df["atom_name"].astype(str).str.strip()
    ))


def map_rcsb_filtered_atoms_to_opm(filtered_rcsb_atoms, opm_atoms):
    """
    Takes atoms filtered using RCSB/Vina coordinate system and returns
    matching atoms from OPM coordinate system.
    """
    filtered_keys = set(atom_match_key(filtered_rcsb_atoms))

    opm_atoms = opm_atoms.copy()
    opm_atoms["_match_key"] = atom_match_key(opm_atoms)

    mapped = opm_atoms[opm_atoms["_match_key"].isin(filtered_keys)].copy()
    mapped = mapped.drop(columns=["_match_key"])

    return mapped


def save_atoms_as_pdb(atom_df, output_path):
    filtered_pdb = PandasPdb()
    filtered_pdb.df["ATOM"] = atom_df.copy()
    filtered_pdb.to_pdb(
        path=str(output_path),
        records=None,
        gz=False,
        append_newline=True
    )


def get_all_opm_clr_gridlists(opm_file):
    ligand = read_clr_atoms(opm_file)
    all_ligands = []

    for residue_number, chain_id in set(zip(ligand["residue_number"], ligand["chain_id"])):
        clr_atoms = ligand[
            (ligand["residue_number"] == residue_number) &
            (ligand["chain_id"] == chain_id)
        ]

        if not clr_atoms.empty:
            all_ligands.append(grid_list(clr_atoms))

    return all_ligands


def check_if_unlabeled_is_positive(positive_grid_list, unlabeled_grid_list, cutoff=5.0):
    positive_grid_list = np.asarray(positive_grid_list, dtype=float)
    unlabeled_grid_list = np.asarray(unlabeled_grid_list, dtype=float)

    if positive_grid_list.size == 0 or unlabeled_grid_list.size == 0:
        return False

    positive_centroid = positive_grid_list.mean(axis=0)
    unlabeled_centroid = unlabeled_grid_list.mean(axis=0)

    distance = np.linalg.norm(positive_centroid - unlabeled_centroid)
    return distance <= cutoff

def make_atom_key(df):
    return list(zip(
        df['residue_number'].astype(int),
        df['chain_id'].astype(str),
        df['atom_name'].astype(str).str.strip()
    ))


def map_rcsb_atoms_to_opm(filtered_rcsb_atoms, opm_protein):
    filtered_keys = set(make_atom_key(filtered_rcsb_atoms))

    opm_protein = opm_protein.copy()
    opm_protein['_key'] = make_atom_key(opm_protein)

    mapped_atoms = opm_protein[opm_protein['_key'].isin(filtered_keys)].copy()
    mapped_atoms = mapped_atoms.drop(columns=['_key'])

    return mapped_atoms

In [10]:
def get_positive_ligand_atoms(positive_file, protein_name):
    protein_pdb_df = PandasPdb().read_pdb(positive_file)
    protein_pdb_df.df.keys()
    protein = protein_pdb_df.df['ATOM']
    protein = protein[~protein['atom_name'].str.startswith('H')] # don't use hydrogen
    protein_coords = protein[['x_coord', 'y_coord', 'z_coord']].values
    protein_centroid = protein_coords.mean(axis=0)
    print(set(protein['chain_id']))
    print(positive_file)

    ligand_df = PandasPdb().read_pdb(positive_file)
    ligand_df.df.keys()
    ligand = ligand_df.df['HETATM']
    ligand = ligand[ligand['residue_name']=="CLR"]
    x = list(set(zip(ligand['residue_number'], ligand['chain_id'])))

    #get the most inward residue
    min_distance = float('inf')
    closest_clr = None

    all_ligands = []

    for residue_number, chain_id in x:
        clr_atoms = ligand[(ligand['residue_number'] == residue_number) & (ligand['chain_id'] == chain_id)]
        if clr_atoms.empty:
            continue

        clr_coords = clr_atoms[['x_coord', 'y_coord', 'z_coord']].values
        clr_centroid = clr_coords.mean(axis=0)
        
        distance = np.linalg.norm(protein_centroid - clr_centroid)
        
        if distance < min_distance:
            min_distance = distance
            closest_clr = (residue_number, chain_id)

        grid_list_ = grid_list(clr_atoms)

        all_ligands.append(grid_list_)

    ligand_ = ligand[(ligand['residue_number'] == closest_clr[0]) & (ligand['chain_id'] == closest_clr[1])]
    grid_list_ = grid_list(ligand_)

    filtered_atoms = filtering_proteins(protein, grid_list_)

    # Save to pdb
    filtered_pdb = PandasPdb()
    filtered_pdb.df['ATOM'] = filtered_atoms
    filtered_pdb_path = f"filtered-opm-ivan-5A/positive/{protein_name}-filtered.pdb"
    os.makedirs(os.path.dirname(filtered_pdb_path), exist_ok=True)
    filtered_pdb.to_pdb(path=filtered_pdb_path, records=None, gz=False, append_newline=True)
    print(f"Saved: {filtered_pdb_path}")

    return protein, all_ligands


In [11]:
def check_if_unlabeled_is_positive(positive_grid_list, unlabeled_grid_list, cutoff=5.0):
    positive_grid_list = np.asarray(positive_grid_list, dtype=float)
    unlabeled_grid_list = np.asarray(unlabeled_grid_list, dtype=float)

    if positive_grid_list.size == 0 or unlabeled_grid_list.size == 0:
        print("Empty positive_grid_list or unlabeled_grid_list")
        return False

    chol_centroid = positive_grid_list.mean(axis=0)
    vina_centroid = unlabeled_grid_list.mean(axis=0)

    distance = np.linalg.norm(chol_centroid - vina_centroid)
    print(f"Centroid distance: {distance:.3f} Å")
    return distance <= cutoff

In [12]:
# positive_files = glob.glob("OPM_PDB/*.pdb")
# positive_files = sorted(positive_files, key=natural_sort_key)

# unlabeled_files = glob.glob("CLR-Unlabeled-Distinct-matched-to-opm_ids/*.pdb")
# unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

# RCSB_PROTEIN_DIR = "CLR-PDB-matched-to-opm_ids"
# OPM_DIR = "OPM_PDB"

# for unlabeled_file in unlabeled_files:
#     unlabeled_name = get_protein_name(unlabeled_file)
#     fragment_index = get_mode_index(unlabeled_file)

#     if unlabeled_name is None:
#         print(f"Skipping file with no PDB id match: {unlabeled_file}")
#         continue

#     rcsb_file = os.path.join(RCSB_PROTEIN_DIR, f"{unlabeled_name}_protein.pdb")
#     opm_file = os.path.join(OPM_DIR, f"{unlabeled_name.lower()}.pdb")

#     if not os.path.exists(rcsb_file):
#         print(f"Missing RCSB file: {rcsb_file}")
#         continue

#     if not os.path.exists(opm_file):
#         print(f"Missing OPM file: {opm_file}")
#         continue

#     # OPM positive logic, keeps closest CLR and all_lig_gridlist
#     opm_protein, all_lig_gridlist = get_positive_ligand_atoms(opm_file, unlabeled_name)

#     # RCSB whole protein for filtering against Vina docked CLR
#     rcsb_pdb = PandasPdb().read_pdb(rcsb_file)
#     rcsb_protein = rcsb_pdb.df['ATOM']
#     rcsb_protein = rcsb_protein[~rcsb_protein['atom_name'].str.startswith('H')]

#     fragment_df = PandasPdb().read_pdb(unlabeled_file)
#     fragment = fragment_df.df['HETATM']
#     #fragment = fragment[fragment['residue_name'] == 'CLR']

#     grid_list_ = grid_list(fragment)

#     # Filter RCSB atoms using Vina CLR coordinates
#     filtered_rcsb_atoms = filtering_proteins(rcsb_protein, grid_list_)

#     if filtered_rcsb_atoms.empty:
#         print(f"No filtered RCSB atoms found for {unlabeled_file}")
#         continue

#     # Convert filtered RCSB atoms to OPM coordinates
#     filtered_opm_atoms = map_rcsb_atoms_to_opm(filtered_rcsb_atoms, opm_protein)

#     if filtered_opm_atoms.empty:
#         print(f"No OPM atom matches found for {unlabeled_file}")
#         continue

#     filtered_pdb = PandasPdb()
#     filtered_pdb.df['ATOM'] = filtered_opm_atoms
#     grid_list_filtered = grid_list(filtered_pdb.df['ATOM'])

#     is_positive = False
#     for lig in all_lig_gridlist:
#         if check_if_unlabeled_is_positive(lig.copy(), grid_list_filtered.copy()):
#             is_positive = True
#             break

#     if is_positive:
#         filtered_pdb_path = f"filtered-opm-vina-5A/unlabeled/{unlabeled_name}-f{fragment_index}-positive.pdb"
#     else:
#         filtered_pdb_path = f"filtered-opm-vina-5A/unlabeled/{unlabeled_name}-f{fragment_index}.pdb"

#     os.makedirs(os.path.dirname(filtered_pdb_path), exist_ok=True)
#     filtered_pdb.to_pdb(path=filtered_pdb_path, records=None, gz=False, append_newline=True)

#     print(f"Saved: {filtered_pdb_path}")

In [13]:
# import glob
# from pathlib import Path

# def pdb_id_from_file(path):
#     """
#     Extract PDB ID from file name.
#     Works for names like:
#     1abc.pdb
#     1ABC.pdb
#     1ABC_protein.pdb
#     1abc_opm.pdb
#     """
#     stem = Path(path).stem
#     stem = stem.replace("_protein", "")
#     return stem[:4].lower()


# # Ivan files are used only to get the PDB IDs
# ivan_files = glob.glob("/home/alexhernandez/p2rank/ivanfiles/*.pdb")

# ivan_pdb_ids = {
#     pdb_id_from_file(f)
#     for f in ivan_files
# }

# print("Number of Ivan PDB IDs:", len(ivan_pdb_ids))


# # Now use OPM PDB files, but only if their PDB ID is in ivanfiles
# opm_files = glob.glob("OPM_PDB/*.pdb")

# positive_files = [
#     f for f in opm_files
#     if pdb_id_from_file(f) in ivan_pdb_ids
# ]

# positive_files = sorted(positive_files, key=natural_sort_key)

# print("Number of matching OPM PDB files:", len(positive_files))


# # Optional: report Ivan IDs that do not have a matching OPM file
# opm_pdb_ids = {
#     pdb_id_from_file(f)
#     for f in opm_files
# }

# missing_from_opm = sorted(ivan_pdb_ids - opm_pdb_ids)

# print("Ivan PDB IDs missing from OPM_PDB:")
# print(missing_from_opm)


# # Run your normal positive logic, but now using matching OPM files
# for positive in positive_files:
#     positive_name = get_protein_name(positive)

#     protein, all_lig_gridlist = get_positive_ligand_atoms(
#         positive,
#         positive_name
#     )

In [14]:
import numpy as np

def compute_inverse_pairwise_distances(df):
    """
    Compute the pairwise Euclidean distances between residues based on their 3D coordinates.

    Parameters:
    df (pd.DataFrame): DataFrame containing 'X', 'Y', 'Z' coordinates and 'NewIndex' as index.

    Returns:
    pd.DataFrame: A DataFrame containing the pairwise distance matrix.
    """
    # Extract the coordinates (X, Y, Z)
    coordinates = df[['X', 'Y', 'Z']].values

    # Calculate pairwise distances using broadcasting
    diff = coordinates[:, np.newaxis, :] - coordinates[np.newaxis, :, :]
    distances = np.sqrt(np.sum(diff ** 2, axis=-1))

    # Compute inverse distance (1/d)
    with np.errstate(divide='ignore'):  # Ignore division by zero warning
        inverse_distances = 1 / distances

    # Set diagonal elements (self-distances) to 1
    np.fill_diagonal(inverse_distances, 1)

    # Cap values at 1
    inverse_distances = np.minimum(inverse_distances, 1)

    return inverse_distances

def pdb_to_dataframe(pdb_file):
    """
    Load a PDB file using MDAnalysis and convert key atom information to a pandas DataFrame.
    """
    u = mda.Universe(pdb_file)
    
    # Extract atom-related data: atom name, residue name, residue ID, and chain ID
    atom_data = {
        'Atom Name': u.atoms.names,
        'Residue Name': u.atoms.resnames,
        'Residue ID': u.atoms.resids,
        'Chain ID': u.atoms.segids,
        'X': u.atoms.positions[:, 0],
        'Y': u.atoms.positions[:, 1],
        'Z': u.atoms.positions[:, 2],
    }
    
    # Create a pandas DataFrame from the atom data
    df = pd.DataFrame(atom_data)
    
    return df

def one_hot_encoding(pdb_df):
    biggest_set = [
        # Carbon (C) subtypes
        'C', 'CA', 'CB', 'CD', 'CD1', 'CD2', 'CE', 'CE1', 'CE2', 'CE3', 'CG', 'CG1', 'CG2', 'CH2', 'CZ', 'CZ2', 'CZ3',

        # Oxygen (O) subtypes
        'O', 'OH', 'OD1', 'OD2', 'OE1', 'OE2', 'OG', 'OG1', 

        # Nitrogen (N) subtypes
        'N', 'NE', 'NE1', 'NE2', 'ND1', 'ND2', 'NZ', 'NH1', 'NH2', 

        # Sulfur (S) subtypes
        'SD', 'SG'
    ]

    biggest_set.append('UNKNOWN')  # Add an additional column for unknown atom types
    
    # Create a zero matrix with shape (num_rows, num_unique_atoms)
    num_rows = len(pdb_df)
    num_cols = len(biggest_set)
    one_hot_matrix = np.zeros((num_rows, num_cols), dtype=int)

    # Create a mapping from atom name to index
    atom_to_index = {atom: idx for idx, atom in enumerate(biggest_set)}

    # Fill the one-hot matrix
    for i, atom in enumerate(pdb_df['Atom Name']):
        if atom in atom_to_index:
            one_hot_matrix[i, atom_to_index[atom]] = 1
        else:
            one_hot_matrix[i, atom_to_index['UNKNOWN']] = 1
            print(atom, "went to unknown column")

    return one_hot_matrix

def compute_z_depth_feature(pdb_df):
    """
    Returns one normalized z-depth value per atom.
    Shape: (num_atoms, 1)
    Uses absolute z distance from membrane center z=0.
    """
    z = pdb_df["Z"].values.astype(float)

    z_depth = np.abs(z)

    max_z = z_depth.max()
    if max_z == 0:
        return np.zeros((len(z_depth), 1))

    z_depth = z_depth / max_z
    return z_depth.reshape(-1, 1)

def min_max_normalization(matrix):
    """
    Perform Min-Max normalization on a given matrix.

    Parameters:
    matrix (np.ndarray): The input matrix to be normalized.

    Returns:
    np.ndarray: The normalized matrix with values scaled to the range [0, 1].
    """
    # Compute the minimum and maximum values for the matrix
    min_val = np.min(matrix)
    max_val = np.max(matrix)

    # Apply Min-Max normalization formula
    normalized_matrix = (matrix - min_val) / (max_val - min_val)

    return normalized_matrix

In [15]:
max_atoms = 150
output_dir = "cholesterol-graph-opm-onehot/positive"
os.makedirs(output_dir, exist_ok=True)

positive_files = glob.glob("filtered-opm-vina-5A/positive/*.pdb")
positive_files = sorted(positive_files, key=natural_sort_key)

for file in positive_files:
    pdb_df = pdb_to_dataframe(file)

    encoded_matrix = one_hot_encoding(pdb_df)
    # inverse_distance = compute_inverse_pairwise_distances(pdb_df)

    # combined_matrix = inverse_distance @ encoded_matrix
    # combined_matrix = min_max_normalization(combined_matrix)

    # z_depth = compute_z_depth_feature(pdb_df)
    # print(z_depth[:10])
    # combined_matrix = combined_matrix * z_depth

    # num_atoms = inverse_distance.shape[0]

    # if num_atoms > max_atoms:
    #     raise Exception(f"{file} has {num_atoms} atoms, exceeding {max_atoms}")

    # combined_matrix = np.pad(
    #     combined_matrix,
    #     ((0, max_atoms - num_atoms), (0, 0)),
    #     mode="constant"
    # )

    base_name = os.path.splitext(os.path.basename(file))[0]
    output_path = os.path.join(output_dir, f"{base_name}_combined_matrix.npy")

    np.save(output_path, encoded_matrix)
    print(f"Saved: {output_path}")

Saved: cholesterol-graph-opm-onehot/positive/1ZHY-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/2RH1-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/2ZXE-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3A3Y-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3AM6-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3D4S-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3N9Y-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3NY8-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3NY9-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3NYA-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3WGU-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/3WGV-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/4DKL-fi

/home/alexhernandez/miniconda3/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:376: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/alexhernandez/miniconda3/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:376: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/alexhernandez/miniconda3/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:376: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to

Saved: cholesterol-graph-opm-onehot/positive/5OLO-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5OLV-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5OLZ-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5OM1-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5OM4-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5TCX-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5TVN-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5UPH-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5UVI-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5VRA-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5WVR-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5X7D-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/5X93-fi

/home/alexhernandez/miniconda3/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:376: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/alexhernandez/miniconda3/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:376: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "
/home/alexhernandez/miniconda3/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:376: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to

Saved: cholesterol-graph-opm-onehot/positive/6KUW-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6LFO-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6LPJ-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6LPK-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6LPL-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6LQG-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6LR4-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6LW5-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6M0F-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6M0Z-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6M2R-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6M3Z-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6M38-fi

/home/alexhernandez/miniconda3/lib/python3.12/site-packages/MDAnalysis/topology/PDBParser.py:376: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "


Saved: cholesterol-graph-opm-onehot/positive/6UD8-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6UY0-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6V22-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6VI4-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6VN7-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6W5S-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6WBF-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6WBG-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6WBM-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6WBN-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6WGT-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6WH4-filtered_combined_matrix.npy
Saved: cholesterol-graph-opm-onehot/positive/6WIV-fi

In [16]:
# max_atoms = 150
# output_dir = "cholesterol-graph-opm-5A/unlabeled"
# os.makedirs(output_dir, exist_ok=True)

# unlabeled_files = glob.glob("filtered-opm-vina-5A/unlabeled/*.pdb")
# unlabeled_files = sorted(unlabeled_files, key=natural_sort_key)

# for file in unlabeled_files:
#     pdb_df = pdb_to_dataframe(file)

#     encoded_matrix = one_hot_encoding(pdb_df)
#     inverse_distance = compute_inverse_pairwise_distances(pdb_df)

#     combined_matrix = inverse_distance @ encoded_matrix
#     combined_matrix = min_max_normalization(combined_matrix)

#     z_depth = compute_z_depth_feature(pdb_df)
#     combined_matrix = combined_matrix * z_depth

#     num_atoms = inverse_distance.shape[0]

#     if num_atoms > max_atoms:
#         # raise Exception(f"{file} has {num_atoms} atoms, exceeding {max_atoms}")
#         continue # Skip files with too many atoms

#     combined_matrix = np.pad(
#         combined_matrix,
#         ((0, max_atoms - num_atoms), (0, 0)),
#         mode="constant"
#     )

#     base_name = os.path.splitext(os.path.basename(file))[0]
#     output_path = os.path.join(output_dir, f"{base_name}_combined_matrix.npy")

#     np.save(output_path, combined_matrix)
#     print(f"Saved: {output_path}")